# Fine-tuning médical QLoRA — Challenge IA TechCorp (volet expérimental)

**Filière IA — branche `groupe-ia-1`** · modèle base : `microsoft/Phi-3.5-mini-instruct` (aligné sur la base propre de prod).

Objectif : fine-tuner un assistant médical **expérimental** (démo, non déployé en prod) en QLoRA 4-bit sur un sous-échantillon de `ruslanmv/ai-medical-chatbot`.

## Comment utiliser ce notebook
1. Ouvrir dans **Google Colab**, menu `Exécution > Modifier le type d'exécution` → **GPU T4** (gratuit).
2. Exécuter les cellules dans l'ordre. Durée totale attendue : ~15-30 min sur T4 (500 exemples, 3 epochs).
3. À la fin : récupérer les **métriques** (loss, epochs, temps) et la **courbe de loss** affichées, et les reporter dans `rendu/ia/`.
4. **Partager le notebook** : `Partager` (haut droite) → *Tous les utilisateurs disposant du lien* → **Lecteur**, puis coller le lien dans `rendu/ia/README.md`.

> ⚠️ Modèle **expérimental** : ne remplace pas un avis médical, non déployé en production (cf. `SUIVI_PROJET.md`).


## 0. Vérifier le GPU


In [ ]:
!nvidia-smi

## 1. Installer les dépendances
Versions épinglées pour la reproductibilité sur Colab (juillet 2026 : ajuster si conflit).


In [ ]:
!pip install -q -U "transformers>=4.44" "trl>=0.9,<0.12" "peft>=0.12" \
    "bitsandbytes>=0.43" "accelerate>=0.33" datasets huggingface_hub matplotlib

## 2. Imports & configuration


In [ ]:
import os, time, random, json
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from datasets import Dataset
from transformers import (AutoModelForCausalLM, AutoTokenizer,
                          BitsAndBytesConfig, TrainingArguments)
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

BASE_MODEL   = "microsoft/Phi-3.5-mini-instruct"
N_SAMPLES    = 500      # sous-echantillon volontairement reduit (demo)
N_EPOCHS     = 3        # 2-3 suffisent pour une demo
MAX_SEQ_LEN  = 1024
OUTPUT_DIR   = "phi35-medical-qlora"
print("torch", torch.__version__, "| CUDA dispo:", torch.cuda.is_available())

## 3. Charger et sous-échantillonner le dataset
Source : [`ruslanmv/ai-medical-chatbot`](https://huggingface.co/datasets/ruslanmv/ai-medical-chatbot) — ~256 916 dialogues, colonnes `Description`, `Patient`, `Doctor`.
On nettoie (doublons, vides, longueurs aberrantes) puis on réduit à 500 exemples reproductibles (seed 42).


In [ ]:
from huggingface_hub import hf_hub_download
path = hf_hub_download('ruslanmv/ai-medical-chatbot', 'dialogues.parquet', repo_type='dataset')
df = pd.read_parquet(path, columns=['Description', 'Patient', 'Doctor'])
print('brut:', len(df), 'lignes')

# --- Nettoyage ---
df = df.dropna(subset=['Patient', 'Doctor'])
df['Patient'] = df['Patient'].str.strip()
df['Doctor']  = df['Doctor'].str.strip()
df = df[(df['Patient'].str.len() > 15) & (df['Doctor'].str.len() > 20)]
df = df[df['Patient'].str.len() < 2000]           # ecarte les questions aberrantes
df = df.drop_duplicates(subset=['Patient'])
print('apres nettoyage:', len(df), 'lignes')

# --- Sous-echantillon reproductible ---
sample = df.sample(n=N_SAMPLES, random_state=SEED).reset_index(drop=True)
sample.to_json('medical_sample_500.jsonl', orient='records', lines=True, force_ascii=False)
print('sous-echantillon:', len(sample), 'exemples -> medical_sample_500.jsonl')
sample.head(2)

## 4. Mise au format chat Phi-3.5
On formate chaque paire en conversation `messages` (system / user / assistant), puis le template applique les balises `<|user|> … <|end|> <|assistant|>`.


In [ ]:
SYSTEM_PROMPT = ('You are a careful medical assistant. Provide helpful, factual health information. '
                 'You are not a doctor: always advise consulting a qualified professional for diagnosis or treatment.')

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.unk_token or tokenizer.eos_token

def to_text(row):
    msgs = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': row['Patient']},
        {'role': 'assistant', 'content': row['Doctor']},
    ]
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)

sample['text'] = sample.apply(to_text, axis=1)
ds = Dataset.from_pandas(sample[['text']])
split = ds.train_test_split(test_size=0.1, seed=SEED)
train_ds, eval_ds = split['train'], split['test']
print('train', len(train_ds), '| eval', len(eval_ds))
print(train_ds[0]['text'][:400])

## 5. Charger Phi-3.5 en 4-bit (QLoRA) + config LoRA


In [ ]:
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, quantization_config=bnb, device_map='auto', trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)
model.config.use_cache = False

lora = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias='none', task_type='CAUSAL_LM',
    target_modules=['qkv_proj', 'o_proj', 'gate_up_proj', 'down_proj'],  # modules Phi-3.5
)

## 6. Entraînement (SFTTrainer) + chronométrage


In [ ]:
cfg = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=N_EPOCHS,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.03,
    logging_steps=10,
    eval_strategy='epoch',
    save_strategy='epoch',
    bf16=True,
    max_seq_length=MAX_SEQ_LEN,
    dataset_text_field='text',
    report_to='none',
    seed=SEED,
)
trainer = SFTTrainer(
    model=model, args=cfg, train_dataset=train_ds, eval_dataset=eval_ds,
    peft_config=lora, processing_class=tokenizer,
)
t0 = time.time()
train_result = trainer.train()
train_time = time.time() - t0
print(f'\n=== Entrainement termine en {train_time/60:.1f} min ===')

## 7. Métriques & courbe de loss (à reporter dans le rendu)


In [ ]:
hist = trainer.state.log_history
train_pts = [(h['epoch'], h['loss']) for h in hist if 'loss' in h]
eval_pts  = [(h['epoch'], h['eval_loss']) for h in hist if 'eval_loss' in h]

metrics = {
    'base_model': BASE_MODEL,
    'n_samples': N_SAMPLES,
    'epochs': N_EPOCHS,
    'train_time_min': round(train_time/60, 2),
    'final_train_loss': round(train_pts[-1][1], 4) if train_pts else None,
    'final_eval_loss': round(eval_pts[-1][1], 4) if eval_pts else None,
}
print(json.dumps(metrics, indent=2))
with open('training_metrics.json', 'w') as f: json.dump(metrics, f, indent=2)

plt.figure(figsize=(8,5))
if train_pts: plt.plot(*zip(*train_pts), label='train loss', marker='.')
if eval_pts:  plt.plot(*zip(*eval_pts), label='eval loss', marker='o')
plt.xlabel('epoch'); plt.ylabel('loss'); plt.title('QLoRA medical fine-tuning — loss')
plt.legend(); plt.grid(alpha=.3); plt.savefig('loss_curve.png', dpi=120, bbox_inches='tight')
plt.show()

## 8. Sauvegarde de l'adapter LoRA


In [ ]:
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print('adapter sauvegarde dans', OUTPUT_DIR)
# Optionnel : telecharger l'adapter + les artefacts
# from google.colab import files; files.download('loss_curve.png'); files.download('training_metrics.json')

## 9. Tests de validation (quelques questions médicales)
Petit contrôle qualitatif du modèle fine-tuné.


In [ ]:
from transformers import pipeline
model.config.use_cache = True
gen = pipeline('text-generation', model=trainer.model, tokenizer=tokenizer)

val_questions = [
    'What are common symptoms of type 2 diabetes?',
    'I have had a mild headache and a runny nose for two days. What could it be?',
    'Is it safe to take ibuprofen and paracetamol together?',
    'What lifestyle changes help lower high blood pressure?',
]
for q in val_questions:
    msgs = [{'role':'system','content':SYSTEM_PROMPT}, {'role':'user','content':q}]
    prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    out = gen(prompt, max_new_tokens=256, do_sample=False, temperature=None, top_p=None)[0]['generated_text']
    print('Q:', q)
    print('A:', out[len(prompt):].strip(), '\n' + '-'*80)

---
### À reporter dans `rendu/ia/`
- Le **lien Colab** (partagé en lecture) → `rendu/ia/README.md`.
- `training_metrics.json` (loss finale, epochs, temps) et `loss_curve.png`.
- Quelques réponses de la cellule 9 comme échantillon de validation.

**Rappel** : modèle médical = expérimental, **pas de déploiement prod**.
